# 04 -- Deployment: FastAPI + Docker + Monitoring
**Project:** Fraud Detection Pipeline | Week 4

Sections: install deps | load model | FastAPI app | test API | Great Expectations | Evidently drift | Dockerfile | Streamlit demo | GitHub Actions | Render deploy | Week 4 summary

## 0. Install dependencies

In [ ]:
import subprocess
packages = [
    'fastapi', 'uvicorn[standard]', 'pydantic',
    'lightgbm', 'xgboost', 'shap',
    'evidently', 'great-expectations',
    'streamlit', 'requests', 'loguru',
    'python-multipart', 'httpx', 'pytest',
]
subprocess.run(['pip', 'install', '-q'] + packages, check=True)
print('All packages ready')

## 1. Load champion model & verify
> Confirm the model and threshold saved in Week 3 load correctly.

In [ ]:
import joblib, json, os
import numpy as np
import pandas as pd

MODEL_PATH     = '../models/lgbm_champion.pkl'
THRESHOLD_PATH = '../models/threshold.json'
FEATURES_PATH  = '../data/processed/feature_cols.json'

assert os.path.exists(MODEL_PATH), f'Model not found: {MODEL_PATH}'
model = joblib.load(MODEL_PATH)
print(f'Model loaded: {type(model).__name__}')

with open(THRESHOLD_PATH) as f:
    threshold_data = json.load(f)
threshold = threshold_data['threshold']
print(f'Threshold: {threshold:.3f}')

with open(FEATURES_PATH) as f:
    feature_cols = json.load(f)
print(f'Features: {len(feature_cols)}')

# Smoke test
dummy = pd.DataFrame(np.zeros((1, len(feature_cols))), columns=feature_cols)
prob  = model.predict_proba(dummy)[0, 1]
print(f'Smoke test score: {prob:.4f}')
print()
print('Model details:')
print(f'  Threshold : {threshold:.3f}')
print(f'  AUC-PR    : {threshold_data.get("auc_pr", "N/A")}')
print(f'  Recall    : {threshold_data.get("recall_at_threshold", "N/A")}')
print('Model is ready to serve')

## 2. Write FastAPI app
> Writes `src/api/main.py` with three endpoints:
> `GET /health` | `GET /model-info` | `POST /predict`

In [ ]:
api_code = (
'import json, os, joblib\n'
'from pathlib import Path\n'
'import numpy as np\n'
'import pandas as pd\n'
'import shap\n'
'from fastapi import FastAPI, HTTPException\n'
'from pydantic import BaseModel, Field\n'
'from typing import Optional\n'
'import uvicorn\n'
'\n'
'app = FastAPI(\n'
'    title="Fraud Detection API",\n'
'    description="Real-time fraud scoring -- LightGBM + SHAP",\n'
'    version="1.0.0"\n'
')\n'
'\n'
'MODEL_PATH     = "models/lgbm_champion.pkl"\n'
'THRESHOLD_PATH = "models/threshold.json"\n'
'FEATURES_PATH  = "data/processed/feature_cols.json"\n'
'\n'
'model = None\n'
'threshold = 0.5\n'
'feature_cols = []\n'
'explainer = None\n'
'\n'
'\n'
'class TransactionRequest(BaseModel):\n'
'    TransactionAmt: float = Field(..., gt=0)\n'
'    ProductCD:      Optional[str]   = None\n'
'    card1:          Optional[int]   = None\n'
'    card4:          Optional[str]   = None\n'
'    card6:          Optional[str]   = None\n'
'    P_emaildomain:  Optional[str]   = None\n'
'    addr1:          Optional[float] = None\n'
'\n'
'\n'
'class FraudPrediction(BaseModel):\n'
'    fraud_probability: float\n'
'    is_fraud:          bool\n'
'    threshold_used:    float\n'
'    risk_level:        str\n'
'    top_features:      dict\n'
'    model_version:     str\n'
'\n'
'\n'
'@app.on_event("startup")\n'
'def load_model():\n'
'    global model, threshold, feature_cols, explainer\n'
'    model = joblib.load(MODEL_PATH)\n'
'    with open(THRESHOLD_PATH) as f:\n'
'        threshold = json.load(f)["threshold"]\n'
'    with open(FEATURES_PATH) as f:\n'
'        feature_cols = json.load(f)\n'
'    explainer = shap.TreeExplainer(model)\n'
'\n'
'\n'
'@app.get("/health")\n'
'def health():\n'
'    return {"status": "ok", "model_loaded": model is not None,\n'
'            "threshold": threshold, "n_features": len(feature_cols)}\n'
'\n'
'\n'
'@app.get("/model-info")\n'
'def model_info():\n'
'    return {"model_type": "LightGBMClassifier",\n'
'            "threshold": threshold, "n_features": len(feature_cols),\n'
'            "version": "1.0.0", "dataset": "IEEE-CIS"}\n'
'\n'
'\n'
'@app.post("/predict", response_model=FraudPrediction)\n'
'def predict(request: TransactionRequest):\n'
'    if model is None:\n'
'        raise HTTPException(status_code=503, detail="Model not loaded")\n'
'    row = {col: 0 for col in feature_cols}\n'
'    for k, v in request.model_dump().items():\n'
'        if k in row and v is not None:\n'
'            row[k] = v\n'
'    df   = pd.DataFrame([row])\n'
'    prob = float(model.predict_proba(df)[0, 1])\n'
'    is_fraud = prob >= threshold\n'
'    risk = "CRITICAL" if prob >= 0.80 else "HIGH" if is_fraud else "MEDIUM" if prob >= 0.20 else "LOW"\n'
'    sv   = explainer.shap_values(df)\n'
'    sv   = sv[1][0] if isinstance(sv, list) else sv[0]\n'
'    top  = np.argsort(np.abs(sv))[-5:][::-1]\n'
'    top_feats = {feature_cols[i]: round(float(sv[i]), 4) for i in top}\n'
'    return FraudPrediction(fraud_probability=round(prob,4),\n'
'        is_fraud=bool(is_fraud), threshold_used=round(threshold,3),\n'
'        risk_level=risk, top_features=top_feats, model_version="lgbm-v1.0")\n'
'\n'
'\n'
'if __name__ == "__main__":\n'
'    uvicorn.run("src.api.main:app", host="0.0.0.0", port=8000, reload=True)\n'
)

os.makedirs('../src/api', exist_ok=True)
with open('../src/api/main.py', 'w') as f:
    f.write(api_code)
print('src/api/main.py written successfully')

## 3. Test API locally
> Starts the API in background and hits all three endpoints.

In [ ]:
import threading, time, requests, sys
sys.path.insert(0, '..')

def run_server():
    import uvicorn
    uvicorn.run('src.api.main:app', host='0.0.0.0', port=8001, log_level='error')

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(5)

BASE = 'http://localhost:8001'

r = requests.get(f'{BASE}/health')
print('GET /health')
print(f'  {r.status_code} -> {r.json()}')
print()

r = requests.get(f'{BASE}/model-info')
print('GET /model-info')
print(f'  {r.status_code} -> {r.json()}')
print()

legit = {'TransactionAmt': 25.0, 'card4': 'visa', 'card6': 'debit'}
r = requests.post(f'{BASE}/predict', json=legit)
res = r.json()
print('POST /predict (small legit transaction)')
print(f'  fraud_probability : {res["fraud_probability"]}')
print(f'  is_fraud          : {res["is_fraud"]}')
print(f'  risk_level        : {res["risk_level"]}')
print()

susp = {'TransactionAmt': 9999.0, 'card4': 'discover'}
r = requests.post(f'{BASE}/predict', json=susp)
res = r.json()
print('POST /predict (large suspicious transaction)')
print(f'  fraud_probability : {res["fraud_probability"]}')
print(f'  is_fraud          : {res["is_fraud"]}')
print(f'  risk_level        : {res["risk_level"]}')
print()

r = requests.post(f'{BASE}/predict', json={'TransactionAmt': -50})
print(f'Invalid input (negative amount) -> status {r.status_code}  <- should be 422')

## 4. Drift monitoring with Evidently AI
> Compare training vs test distributions.
> Generates a shareable HTML report — commit to GitHub.

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.metrics import DatasetDriftMetric, DatasetMissingValuesMetric

X_train = pd.read_parquet('../data/processed/X_train.parquet')
X_test  = pd.read_parquet('../data/processed/X_test.parquet')

num_cols = X_train.select_dtypes(include='number').columns[:30].tolist()
ref  = X_train[num_cols].sample(n=5000, random_state=42)
curr = X_test[num_cols].sample(n=2000, random_state=42)

print(f'Reference (train) : {ref.shape}')
print(f'Current   (test)  : {curr.shape}')
print('Generating drift report...')

report = Report(metrics=[
    DatasetDriftMetric(),
    DatasetMissingValuesMetric(),
    DataDriftPreset(),
])
report.run(reference_data=ref, current_data=curr)

os.makedirs('../data/external', exist_ok=True)
report.save_html('../data/external/drift_report.html')
print('Saved: data/external/drift_report.html')

result_dict  = report.as_dict()
drift_metric = result_dict['metrics'][0]
print()
print('=== Drift Summary ===')
print(f'Dataset drift   : {drift_metric["result"].get("dataset_drift", "N/A")}')
print(f'Drifted features: {drift_metric["result"].get("number_of_drifted_columns", "N/A")}')
print(f'Total checked   : {drift_metric["result"].get("number_of_columns", "N/A")}')

## 5. Write Dockerfile & render.yaml
> Container definition for deploying to Render.com.

In [ ]:
dockerfile_lines = [
    'FROM python:3.11-slim',
    '',
    'WORKDIR /app',
    '',
    'RUN apt-get update && apt-get install -y libgomp1 && rm -rf /var/lib/apt/lists/*',
    '',
    'COPY requirements.txt .',
    'RUN pip install --no-cache-dir -r requirements.txt',
    '',
    'COPY src/ ./src/',
    'COPY configs/ ./configs/',
    'COPY models/ ./models/',
    'COPY data/processed/feature_cols.json ./data/processed/feature_cols.json',
    '',
    'EXPOSE 8000',
    '',
    'CMD ["uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"]',
]

os.makedirs('../docker', exist_ok=True)
with open('../docker/Dockerfile', 'w') as f:
    f.write('\n'.join(dockerfile_lines))
print('docker/Dockerfile written')

render_lines = [
    'services:',
    '  - type: web',
    '    name: fraud-detection-api',
    '    env: docker',
    '    dockerfilePath: ./docker/Dockerfile',
    '    plan: free',
    '    healthCheckPath: /health',
]
with open('../render.yaml', 'w') as f:
    f.write('\n'.join(render_lines))
print('render.yaml written')
print()
print('Local docker test commands:')
print('  cd ..')
print('  docker build -f docker/Dockerfile -t fraud-api .')
print('  docker run -p 8000:8000 fraud-api')
print('  open http://localhost:8000/docs')

## 6. Write Streamlit demo app
> Visual front-end anyone can use to test the API.
> Deploy free to Hugging Face Spaces.

In [ ]:
streamlit_lines = [
    'import streamlit as st',
    'import requests',
    '',
    'API_URL = "https://fraud-detection-api.onrender.com"',
    '',
    'st.set_page_config(page_title="Fraud Detection Demo", page_icon="?", layout="centered")',
    'st.title("Real-Time Fraud Detection")',
    'st.markdown("Enter transaction details. Model scores fraud probability using **LightGBM + SHAP**.")',
    'st.divider()',
    '',
    'col1, col2 = st.columns(2)',
    'with col1:',
    '    amount   = st.number_input("Transaction Amount ($)", min_value=0.01, max_value=50000.0, value=250.0)',
    '    card_type= st.selectbox("Card Type", ["visa","mastercard","discover","american express"])',
    'with col2:',
    '    card_cat  = st.selectbox("Card Category", ["debit","credit","charge card"])',
    '    email_dom = st.selectbox("Email Domain", ["gmail.com","yahoo.com","outlook.com","anonymous.com"])',
    '',
    'if st.button("Analyse Transaction", type="primary", use_container_width=True):',
    '    payload = {"TransactionAmt": amount, "card4": card_type,',
    '               "card6": card_cat, "P_emaildomain": email_dom}',
    '    with st.spinner("Scoring..."):',
    '        try:',
    '            r   = requests.post(f"{API_URL}/predict", json=payload, timeout=15)',
    '            res = r.json()',
    '            prob     = res["fraud_probability"]',
    '            is_fraud = res["is_fraud"]',
    '            risk     = res["risk_level"]',
    '            if is_fraud:',
    '                st.error(f"FRAUD DETECTED | Risk: {risk}")',
    '            else:',
    '                st.success(f"Transaction Cleared | Risk: {risk}")',
    '            m1, m2, m3 = st.columns(3)',
    '            m1.metric("Fraud Probability", f"{prob:.1%}")',
    '            m2.metric("Decision", "BLOCK" if is_fraud else "ALLOW")',
    '            m3.metric("Risk Level", risk)',
    '            st.subheader("Why this decision? (SHAP)")',
    '            for feat, val in res["top_features"].items():',
    '                direction = "pushed toward fraud" if val > 0 else "reduced fraud score"',
    '                st.write(f"**{feat}** ({val:+.4f}) -- {direction}")',
    '        except Exception as e:',
    '            st.error(f"Error: {e}")',
    '',
    'st.divider()',
    'st.caption("LightGBM | IEEE-CIS 590k transactions | AUC-PR 0.86 | Rakesh Kumar Dubey")',
]

os.makedirs('../streamlit_app', exist_ok=True)
with open('../streamlit_app/app.py', 'w') as f:
    f.write('\n'.join(streamlit_lines))

with open('../streamlit_app/requirements.txt', 'w') as f:
    f.write('streamlit\nrequests\n')

print('streamlit_app/app.py written')
print('streamlit_app/requirements.txt written')
print()
print('Local test: cd ../streamlit_app && streamlit run app.py')

## 7. GitHub Actions CI/CD
> Runs pytest + ruff lint on every push to main.

In [ ]:
ci_lines = [
    'name: CI',
    '',
    'on:',
    '  push:',
    '    branches: [main, develop]',
    '  pull_request:',
    '    branches: [main]',
    '',
    'jobs:',
    '  test:',
    '    runs-on: ubuntu-latest',
    '    steps:',
    '      - uses: actions/checkout@v4',
    '',
    '      - name: Set up Python',
    '        uses: actions/setup-python@v5',
    '        with:',
    '          python-version: "3.11"',
    '          cache: "pip"',
    '',
    '      - name: Install dependencies',
    '        run: pip install -r requirements-dev.txt',
    '',
    '      - name: Lint with ruff',
    '        run: ruff check src/ tests/ --ignore E501',
    '',
    '      - name: Run tests',
    '        run: pytest tests/ -v --cov=src --cov-report=term-missing',
]

os.makedirs('../.github/workflows', exist_ok=True)
with open('../.github/workflows/ci.yml', 'w') as f:
    f.write('\n'.join(ci_lines))
print('.github/workflows/ci.yml written')
print()
print('CI badge for README.md:')
print('[![CI](https://github.com/rakeshdubey1504/fraud-detection/actions/workflows/ci.yml/badge.svg)]'
      '(https://github.com/rakeshdubey1504/fraud-detection/actions/workflows/ci.yml)')

## 8. Deployment steps

In [ ]:
steps = [
    '=== RENDER.COM DEPLOYMENT ===',
    '',
    '1. Push all files to GitHub:',
    '   git add .',
    '   git commit -m "feat: Week 4 -- FastAPI + Docker + Streamlit"',
    '   git push origin main',
    '',
    '2. Go to render.com -> sign up with GitHub (free)',
    '',
    '3. New + -> Web Service -> connect fraud-detection repo',
    '   Name        : fraud-detection-api',
    '   Environment : Docker',
    '   Dockerfile  : ./docker/Dockerfile',
    '   Plan        : Free',
    '   -> Deploy',
    '',
    '4. Live URL after ~5 min:',
    '   https://fraud-detection-api.onrender.com/docs',
    '',
    '5. Update API_URL in streamlit_app/app.py with your Render URL',
    '',
    '=== HUGGING FACE SPACES DEPLOYMENT ===',
    '',
    '1. Go to huggingface.co/spaces',
    '2. New Space -> Streamlit -> name: fraud-detection-demo',
    '3. Upload streamlit_app/app.py and streamlit_app/requirements.txt',
    '4. Live in ~2 minutes:',
    '   https://huggingface.co/spaces/rakeshdubey1504/fraud-detection-demo',
]
for line in steps:
    print(line)

## 9. Week 4 summary & checklist

In [ ]:
print('=' * 58)
print('  WEEK 4 -- DEPLOYMENT COMPLETE')
print('=' * 58)
print()
print('Files written this week:')
print('  src/api/main.py                   FastAPI app')
print('  docker/Dockerfile                 container')
print('  render.yaml                       Render config')
print('  streamlit_app/app.py              demo UI')
print('  streamlit_app/requirements.txt')
print('  .github/workflows/ci.yml          CI pipeline')
print('  data/external/drift_report.html   Evidently report')
print()
print('Checklist before Week 4 LinkedIn post:')
print('  [ ] FastAPI /predict returns fraud score locally')
print('  [ ] Docker container builds and runs locally')
print('  [ ] Render URL is live')
print('  [ ] Streamlit demo live on HF Spaces')
print('  [ ] GitHub Actions shows green CI badge')
print('  [ ] drift_report.html committed to GitHub')
print('  [ ] README updated with all live URLs')
print()
print('Your live URLs:')
print('  API    : https://fraud-detection-api.onrender.com/docs')
print('  Demo   : https://huggingface.co/spaces/rakeshdubey1504/fraud-detection-demo')
print('  MLflow : https://dagshub.com/rakeshdubey1504/fraud-detection.mlflow')
print('  GitHub : https://github.com/rakeshdubey1504/fraud-detection')
print()
print('NEXT -> Week 5: Kafka streaming + graph-based fraud detection')
print('=' * 58)